## Aeropulse — Gold: Date Dimension

**Purpose:** Generates `dim_date` as a standalone calendar table (2018-01-01 to 2027-12-31) rather than deriving it from a source table — the standard approach for a date dimension. `date_id` (`yyyyMMdd`) is the join key `fact_flight.flight_date_id` uses.

**Not batch-parameterised** — deterministic and source-independent; a plain overwrite each run is correct and doesn't need incremental/merge handling.

**Writes:** `dim_date` (full overwrite)


In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

StatementMeta(, 55c7f933-97b3-4467-993a-388bf620ce62, 8, Finished, Available, Finished, False)

In [8]:
start_date, end_date = "2018-01-01", "2027-12-31"   # wide enough to cover future batches without re-running this

dim_date_df = (
    spark.createDataFrame([(1,)], ["_"])
    .withColumn("full_date", F.explode(F.sequence(
        F.to_date(F.lit(start_date)), F.to_date(F.lit(end_date)), F.expr("interval 1 day")
    )))
    .drop("_")
    .withColumn("date_id", F.date_format("full_date", "yyyyMMdd").cast("int")) 
    .withColumn("year", F.year("full_date"))
    .withColumn("quarter", F.quarter("full_date"))
    .withColumn("month", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("day_of_month", F.dayofmonth("full_date"))
    .withColumn("day_of_week", F.dayofweek("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("is_weekend", F.col("day_of_week").isin(1, 7))
)


dim_date_df.write.format("delta").mode("overwrite").saveAsTable("dim_date")

StatementMeta(, 55c7f933-97b3-4467-993a-388bf620ce62, 10, Finished, Available, Finished, False)